# PyTorch Mini Lab — Every Line Explained

**What this lab does, in one sentence:** it teaches a computer to look at a photo of a pet and guess which of 37 breeds it is.

**The 3 big parts:**

1. **Build the network** — the 'brain' that looks at the photo.
2. **Train it** — show it many labelled photos so it learns and fixes its mistakes.
3. **Test it** — show it new photos it has never seen and count how many it gets right.


In [ ]:
import torch  # Load PyTorch, the main tool for building and training neural networks
import torch.nn as nn  # Load the part of PyTorch for building network layers; 'nn' is just a short nickname for it
import torchvision  # Load the part of PyTorch for ready-made photo datasets, like the pet photos used here
import torchvision.transforms as transforms  # Load the tools that change photos before training, e.g. turn them into numbers
from torch.utils.data import Dataset, DataLoader  # Load DataLoader (used below to serve photos in small groups). Dataset is loaded here too, but this lab never actually uses it.

### Network Definition

**Big idea, in simple English:** a photo goes in, and the network slowly shrinks it while learning more and more details about it.

- The photo starts big in size (256 x 256) but simple: only 3 colour layers (red, green, blue).
- Each step makes the picture smaller, but adds more 'feature' layers — like going from 'there is an edge here' to 'there is an ear here'.
- At the end, the network flattens everything into one long list of numbers and picks 1 of 37 breeds.

Think of it like zooming out: you lose small details of where things are, but you gain a better idea of _what_ the whole thing is.


In [ ]:
# This network is too shallow for the task. Your job is to make it deeper!
# 'Shallow' means it has only a few layers. Deeper usually means it can learn more complex patterns.
# Plan in the original notes: go from something big in size (256) with few features (3) to something
# smaller in size (64) with more features (128), then flatten it and guess 1 of 37 breeds.
#
# Moving from something big spatially 256, with few features 3 (depth) to something smaller spatially 64,
# with more features 128 (depth) and then flattening it to a vector of size 128*64*64 = 524288 and then
# passing it through a fully connected layer to get the final output of size 37 (number of classes)

# In PyTorch's nn.Conv2d(3, 64, 3, padding=1), the numbers represent in_channels=3, out_channels=64, kernel_size=3, and padding=1. Together, these parameters define a convolutional layer that takes a 3-channel image (such as RGB), applies 64 distinct \(3 \times 3\) filters, and preserves the input's spatial height and width.

# Parameter Breakdown

# • 3 (in_channels): The depth of the input tensor. For a standard color image, this represents the 3 color channels (Red, Green, and Blue). Each filter in this layer will have a depth of 3 to process all input channels simultaneously.
# • 64 (out_channels): The number of distinct filters (kernels) the layer will learn. Each filter scans the input to detect a specific feature (like edges or textures) and produces one 2D feature map, resulting in an output tensor with 64 channels.
# • 3 (kernel_size): The spatial dimensions of each sliding filter window, meaning each filter is \(3 \times 3\) pixels wide and tall (and 3 channels deep, making each individual filter volume \(3 \times 3 \times 3\)).
# • padding=1: Adds a 1-pixel border of zeros around the outer edges of the input image. With a \(3 \times 3\) kernel and a default stride of 1, padding of 1 ensures the output feature maps maintain the exact same spatial height and width as the input (known as "same" padding).


# class MyNetwork(nn.Module):  # Create your own network type called MyNetwork. It inherits from nn.Module, which is PyTorch's base for all networks.
#     def __init__(self):  # __init__ runs once when the network is created. This is where you list all its layers.
#         super(MyNetwork, self).__init__()  # Set up the parent nn.Module first, so PyTorch can track the layers you add.
#         self.conv1 = nn.Conv2d(3, 64, 3, padding=1)  # Layer 1, a convolution: look at each 3x3 patch of the photo. Input: 3 colours. Output: 64 feature maps. Size stays 256x256 because padding=1 keeps the edges.
#         self.pool1 = nn.MaxPool2d(2, 2)  # Shrink the picture by half: keep only the biggest value in each 2x2 square. Size: 256x256 -> 128x128. The 64 feature layers stay 64.
#         self.ReLU = nn.ReLU()  # The activation function: turn every negative number into 0, keep positive numbers as they are. This lets the network learn non-straight-line patterns.
#         self.conv2 = nn.Conv2d(64, 128, 3, padding=1)  # Layer 2, a convolution: input is the 64 feature maps from before. Output: 128 feature maps. Size stays 128x128.
#         self.pool2 = nn.AvgPool2d(2, 2)  # Shrink again by half, this time by taking the average of each 2x2 square. Size: 128x128 -> 64x64. The 128 feature layers stay 128.
#         self.fc1 = nn.Linear(128*64*64, 37)  # The final layer: take the long flattened list of 128*64*64 = 524288 numbers and turn it into 37 scores, one score for each breed.

#         # To add layers, the size of the input to the next layer must match the output of the previous layer.
#         # For example, if you add another convolutional layer after conv2, its input must be 128 channels, size 64x64.
#         # If you add a fully connected layer after fc1, its input must be 37, because fc1 outputs 37 numbers.
#         # For the linear layer, the input size is 128*64*64 because the output of the previous layer is 64x64x128,
#         # which is flattened to a vector of size 128*64*64 before being passed to the linear layer.
# def forward(self, x):  # forward() says what happens to a photo (x) when it passes through the network, step by step.
#         x = self.ReLU(self.conv1(x))  # Step 1: run the first convolution, then ReLU. Now x is 256x256 with 64 features.
#         x = self.pool1(x)  # Step 2: max-pool to shrink it. Now x is 128x128 with 64 features.
#         x = self.ReLU(self.conv2(x))  # Step 3: run the second convolution, then ReLU. Now x is 128x128 with 128 features.
#         x = self.pool2(x)  # Step 4: average-pool to shrink it again. Now x is 64x64 with 128 features.
#         x = self.fc1(torch.flatten(x, start_dim=1))  # Step 5: flatten each photo into one long list (keep the batch separate with start_dim=1), then run the final layer to get 37 breed scores.
#         return x  # Send the 37 scores back as the network's answer for this photo.


####------------ ASSIGNMENT--------------- ####
# Modify the code inside MyNetwork to change the design of the neural network. Specifically, modify the convolution and pooling layers to be the following:
# 5x5x64 Convolution
# 3x3x128 Convolution
# 2x2 MaxPool
# 3x3x256 Convolution
# 3x3x256 Convolution
# 2x2 MaxPool
# 3x3x512 Convolution
# 2x2 MaxPool
# 3x3x512 Convolution
# Linear Layer (the input size should be changed to the final size of your images, which start as 256x256x3)

class MyNetwork(nn.Module):
    def __init__(self):
        super(MyNetwork, self).__init__()
        self.ReLU = nn.ReLU() #remember
        self.conv1 = nn.Conv2d(3, 64, 5, padding=2)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.conv4 = nn.Conv2d(256, 256, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv5 = nn.Conv2d(256, 512, 3, padding=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.conv6 = nn.Conv2d(512, 512, 3, padding=1)
        self.fc1 = nn.Linear(512*32*32, 37) #fc means fully connected. The 1 means it is the first, and here only, fully connected layer.
        # The first number, 512*32*32 = 524288, is how many numbers come in. That is the photo after the last convolution, flattened into one long list: 32 wide x 32 tall x 512 features.
        # The second number, 37, is how many scores go out, one score for each of the 37 pet breeds.
        # Fully connected means every one of those 524288 numbers is connected to every one of the 37 outputs. The network learns a weight for each connection, then the breed with the highest score is its guess.


#this function determines the order in which the operations defined above are applied to the input data (x). It specifies how the input image is processed through the network layers, including convolutional layers, activation functions, and pooling layers, ultimately leading to the final output of breed scores.
    def forward(self, x):
        x = self.ReLU(self.conv1(x))
        x = self.ReLU(self.conv2(x))
        x = self.pool1(x)
        x = self.ReLU(self.conv3(x))
        x = self.ReLU(self.conv4(x))
        x = self.pool2(x)
        x = self.ReLU(self.conv5(x))
        x = self.pool3(x)
        x = self.ReLU(self.conv6(x))
        x = self.fc1(torch.flatten(x, start_dim=1))
        return x

    
#######################  # A divider line, just to separate the network definition from the line below. It does nothing.

model = MyNetwork()  # Actually build one network from the plan above and call it 'model'. From now on, model is the brain we train.

### Dataset and Training Loop

**Big idea, in simple English:**

- **Dataset:** the Oxford IIIT Pet photos — pictures of cats and dogs, labelled with their breed (37 breeds in total).
- **Transform:** before training, every photo is changed in the same way: turned into numbers, cropped to 256 x 256, and rescaled.
- **DataLoader:** serves the photos to the network in small groups of 8, called batches. Learning in small groups is faster and steadier than one photo at a time.
- **Training loop:** for each photo group, the network guesses, we measure how wrong it was (the loss), and we nudge its weights a tiny bit to do better next time. Repeat 5 times over the whole dataset.


In [ ]:
transform = transforms.Compose([transforms.ToTensor(),  # Step 1 of the photo preparation: turn the photo (pixels 0-255) into a PyTorch tensor (numbers 0-1), with colour first: 3 x 256 x 256.
                                transforms.RandomCrop((256, 256), pad_if_needed=True),  # Step 2: cut a random 256x256 square out of the photo. Pad with extra pixels first if the photo is smaller than that.
                                transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])  # Step 3: shift the numbers so they sit around 0 (roughly -1 to +1). This helps the network learn faster.

train_dataset = torchvision.datasets.OxfordIIITPet("pets_data/", "trainval", transform=transform, download=True)  # Load the training photos: the 'trainval' split (train + validation photos). Save them in the folder pets_data/, apply the transform above, and download them if they are not there yet.
test_dataset = torchvision.datasets.OxfordIIITPet("pets_data/", "test", transform=transform, download=True)  # Load the test photos: the separate 'test' split the network has never seen. Same folder, same transform, download if needed.

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)  # Make a loader for training: give 8 photos at a time, and shuffle the order each time so the network does not memorise the order.
test_loader = DataLoader(test_dataset, batch_size=8)  # Make a loader for testing: also 8 photos at a time, but no shuffle (order does not matter for testing).


learning_rate = 1e-4  # How big each learning step is: 1e-4 means 0.0001. Small steps learn slowly but safely; big steps learn fast but can overshoot.
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)  # Create the optimizer (Adam): the part that changes the network's weights to reduce the loss. It is told to update all of model's weights, using the learning rate above.
loss_func = torch.nn.CrossEntropyLoss()  # Create the loss function: it measures how wrong the network's guess was. Big loss = very wrong. It is made for picking 1 class out of many (37 breeds).

num_epochs = 5  # An epoch is one full pass through all the training photos. Train for 5 full passes.

for epoch in range(num_epochs):  # Start the outer loop: do everything inside it once per epoch, with epoch = 0, 1, 2, 3, 4.
    print("Epoch:", epoch)  # Print which pass we are on, so you can follow the training.
    for im, label, in train_loader:  # Start the inner loop: take one batch of 8 photos at a time. 'im' is the 8 photos, 'label' is their 8 true breed numbers. (The comma after label is harmless.)

        optimizer.zero_grad()  # Clear out the old gradients (the 'how to change' notes) from the last batch, so they do not pile up.

        output = model(im)  # Forward pass: feed the 8 photos through the network and get 8 sets of 37 breed scores.
        loss = loss_func(output, label)  # Compare the network's scores with the true labels and get one number: how wrong it was on this batch.
        print(loss)  # Print that loss number, so you can watch it (hopefully) go down as training goes on.

        loss.backward()  # Backward pass: work out, for every weight in the network, how much it contributed to the mistake. This is called backpropagation.
        optimizer.step()  # Update step: change every weight a tiny bit in the direction that would make the loss smaller.

### Testing

**Big idea, in simple English:** now we stop learning and just check the work. We show the network new pet photos, take its best guess for each one, and count how many guesses match the true breed. That share, as a percentage, is the accuracy.


In [ ]:
correct = 0; total = 0  # Start two counters at zero: how many photos it got right, and how many photos it has seen in total. (Two short lines joined by a semicolon.)

model.eval()  # Switch the network to test mode: no learning, and layers like dropout/batch-norm (none are used here, but it is good habit) behave differently.
with torch.no_grad():  # Do everything inside this block without tracking gradients: we are not learning, so this saves memory and time.
    for im, label in test_loader:  # Go through the test photos in batches of 8. 'im' is the photos, 'label' is their true breeds.
        
        output = model(im)  # Feed the batch through the network and get 37 breed scores for each photo.
        guess = torch.argmax(output, dim=1)  # For each photo, pick the breed with the highest score. argmax means 'the position of the biggest number'.
        correct += torch.sum(guess == label).item()  # Count how many guesses in this batch match the true labels, and add that to the correct counter. .item() turns the PyTorch number into a plain Python number.
        total += len(guess)  # Add the number of photos in this batch (8) to the total counter.

print("Testing Accuracy is", correct / total * 100, "%")  # Print the final score: correct divided by total, times 100, as a percentage.